# Import libraries

In [ ]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [2]:
from src.data import dataloader
from src.data.io import load_dataset, save_dataset
from src.utils.memory import print_memory_stats
from src.trees import fastodt
import numpy as np
import pandas as pd
import time

# Old data splits

In [ ]:
data_dir = '../data'

lag_args = {
    #'lag_5m':'5M',
    'lag_1h':'1H',
    'lag_24h':'24H',
    'lag_168h':'168H'} # 24*7 = 168h, one week ago

# list feature names
columns_to_add_lag = [
            'total_power_active', 'total_power_reactive', 'total_power_apparent',
            'import_energy_active', 'import_energy_reactive', 'export_energy_reactive',
            'total_energy_active', 'total_energy_reactive'
        ]

# target
target_feature = 'total_power_active'

# list of devices to be used
device_list = [
    "f90e7387-9810-408d-885b-6f3841f4ec3f",
    "dd40019b-5b35-4fba-b0dd-16083c1fdb84",
    "3617dc75-38d4-4875-b66c-64f896b5c9f1",
    "a4b80cd7-5438-44cd-8257-c9ac81684822",
    "522e56a5-ff0c-4ece-923f-a925aabe15e6",
    "4a448440-d2dd-4b27-ae02-87e7e3baa9f5",
    "fca96b30-ca2f-41b8-b987-3207af296b03",
    "4ad3e3e6-5f11-4923-a88d-668b4ddb493a",
    ]

In [ ]:
# Start timer to measure processing time
start_time = time.time()

# Generate data with optimized memory usage
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save_intermediate=False,  # Save processed data for faster future loading
    intermediate_prefix='8_dev_hour_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
)

# Calculate and print the processing time
processing_time = time.time() - start_time
print(f"Data processing completed in {(processing_time/60):.2f} minutes")

[INFO] Aggregating data by: 1H
input_features:  ['device_id', 'time', 'export_energy_active', 'hour', 'weekday', 'month', 'total_power_active_lag_1h', 'total_power_active_lag_24h', 'total_power_active_lag_168h', 'total_power_reactive_lag_1h', 'total_power_reactive_lag_24h', 'total_power_reactive_lag_168h', 'total_power_apparent_lag_1h', 'total_power_apparent_lag_24h', 'total_power_apparent_lag_168h', 'import_energy_active_lag_1h', 'import_energy_active_lag_24h', 'import_energy_active_lag_168h', 'import_energy_reactive_lag_1h', 'import_energy_reactive_lag_24h', 'import_energy_reactive_lag_168h', 'export_energy_reactive_lag_1h', 'export_energy_reactive_lag_24h', 'export_energy_reactive_lag_168h', 'total_energy_active_lag_1h', 'total_energy_active_lag_24h', 'total_energy_active_lag_168h', 'total_energy_reactive_lag_1h', 'total_energy_reactive_lag_24h', 'total_energy_reactive_lag_168h']

input_features:  ['device_id', 'time', 'export_energy_active', 'hour', 'weekday', 'month', 'total_power

```python
Overall data time range:  2024-01-07 23:00:02.287000 2024-12-31 22:51:01.931000

Test data time range:  2024-10-20 12:15:18.218000 2024-12-31 22:51:01.931000

(Train + Valid) data range:  2024-01-07 23:00:02.287000 2024-10-20 12:15:16.553000
Split 1:
  Train window: 2024-01-07 23:00:02.287000 to 2024-02-03 23:00:01.625324222
  Valid window: 2024-02-03 23:00:10.155160515 to 2024-03-02 22:59:59.675462482

Split 2:
  Train window: 2024-01-07 23:00:02.287000 to 2024-03-02 22:59:59.675462482
  Valid window: 2024-03-02 23:00:08.205298776 to 2024-03-30 22:59:57.725600743

Split 3:
  Train window: 2024-01-07 23:00:02.287000 to 2024-03-30 22:59:57.725600743
  Valid window: 2024-03-30 23:00:06.255437036 to 2024-04-27 22:59:55.775739004

Split 4:
  Train window: 2024-01-07 23:00:02.287000 to 2024-04-27 22:59:55.775739004
  Valid window: 2024-04-27 23:00:04.305575296 to 2024-05-25 22:59:53.825877264

Split 5:
  Train window: 2024-01-07 23:00:02.287000 to 2024-05-25 22:59:53.825877264
  Valid window: 2024-05-25 23:00:02.355713558 to 2024-06-22 23:00:00.405851818

Split 6:
  Train window: 2024-01-07 23:00:02.287000 to 2024-06-22 23:00:00.405851818
  Valid window: 2024-06-22 23:00:08.935688112 to 2024-07-20 22:59:58.455990078

Split 7:
  Train window: 2024-01-07 23:00:02.287000 to 2024-07-20 22:59:58.455990078
  Valid window: 2024-07-20 23:00:06.985826372 to 2024-08-17 22:59:56.506128340

Split 8:
  Train window: 2024-01-07 23:00:02.287000 to 2024-08-17 22:59:56.506128340
  Valid window: 2024-08-17 23:00:05.035964632 to 2024-09-14 22:59:54.556266600

Split 9:
  Train window: 2024-01-07 23:00:02.287000 to 2024-09-14 22:59:54.556266600
  Valid window: 2024-09-14 23:00:03.086102892 to 2024-10-12 23:00:01.136241156

Split 10:
  Train window: 2024-01-07 23:00:02.287000 to 2024-10-12 23:00:01.136241156
  Valid window: 2024-10-12 23:00:09.666077448 to 2024-10-20 12:15:16.553000
```

In [ ]:
print_memory_stats(train, valid, test)

Memory usage:
  Train: 0.00 MB
  Valid: 0.00 MB
  Test:  0.00 MB
  Total: 0.00 MB


## Additional data save if `save_intermediate` is `False` in `get_data_splits`

In [ ]:
# Save data in optimized NumPy format
train_path, valid_path, test_path = save_dataset(
    train=train,
    valid=valid,
    test=test,
    data_dir=data_dir,
    prefix='5_min_wfv',  # Custom prefix for the files
    use_numpy=True  # Use NumPy's compressed format
)

print(f"Data saved to:\n{train_path}\n{valid_path}\n{test_path}")

# Print memory usage statistics
print_memory_stats(train, valid, test)

## Load data code

In [ ]:
# Measure loading time
start_time = time.time()

# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = load_dataset(data_dir=data_dir, prefix='8_dev_5_min_wfv', use_numpy=True)

loading_time = time.time() - start_time
print(f"Data loaded in {loading_time/60:.2f} minutes")

# New data splits

In [ ]:
# If you're using Colab, connect to google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
lag_args = {
    #'lag_5m':'5M',
    'lag_1h':'1H',
    'lag_24h':'24H',
    }
#data_dir = "/content/drive/My Drive/extracted_files/first_batch"
data_dir = '../data/first_batch'

columns_to_add_lag = [
            'L1_act_power', 'L1_app_power', 'L1_current', 'L1_react_power', 'L1_voltage',
            'L2_act_power', 'L2_app_power', 'L2_current', 'L2_react_power', 'L2_voltage', 
            'L3_act_power', 'L3_app_power', 'L3_current', 'L3_react_power', 'L3_voltage', 
            'grid_freq', 'powerfactor1', 'powerfactor2', 'powerfactor3'
        ]
device_list = ['df_TRB145-4', 
               #'df_TRB145-5',
               ]

# target
target_feature = 'L2_act_power'

In [4]:
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data for faster future loading
    intermediate_prefix='dev_4_hourly_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
    new_data=True,
    fill_missing=True
)

[INFO] Aggregating data by: 1H
input_features:  ['time', 'device_id', 'hour', 'weekday', 'month', 'L1_act_power_lag_1h', 'L1_act_power_lag_24h', 'L1_app_power_lag_1h', 'L1_app_power_lag_24h', 'L1_current_lag_1h', 'L1_current_lag_24h', 'L1_react_power_lag_1h', 'L1_react_power_lag_24h', 'L1_voltage_lag_1h', 'L1_voltage_lag_24h', 'L2_act_power_lag_1h', 'L2_act_power_lag_24h', 'L2_app_power_lag_1h', 'L2_app_power_lag_24h', 'L2_current_lag_1h', 'L2_current_lag_24h', 'L2_react_power_lag_1h', 'L2_react_power_lag_24h', 'L2_voltage_lag_1h', 'L2_voltage_lag_24h', 'L3_act_power_lag_1h', 'L3_act_power_lag_24h', 'L3_app_power_lag_1h', 'L3_app_power_lag_24h', 'L3_current_lag_1h', 'L3_current_lag_24h', 'L3_react_power_lag_1h', 'L3_react_power_lag_24h', 'L3_voltage_lag_1h', 'L3_voltage_lag_24h', 'grid_freq_lag_1h', 'grid_freq_lag_24h', 'powerfactor1_lag_1h', 'powerfactor1_lag_24h', 'powerfactor2_lag_1h', 'powerfactor2_lag_24h', 'powerfactor3_lag_1h', 'powerfactor3_lag_24h']

Overall data time range:  

# Playground
Here I was just double-checking things and playing around with the data. You can easily delete everything after this part of the notebook.

In [ ]:
df1 = pd.DataFrame(
    train[0][0],
    columns=['time', 'device_id', 'hour', 'weekday', 'month', 'total_power_active_lag_5m', 'total_power_active_lag_1h', 'total_power_active_lag_24h', 'total_power_active_lag_168h', 'total_power_reactive_lag_5m', 'total_power_reactive_lag_1h', 'total_power_reactive_lag_24h', 'total_power_reactive_lag_168h', 'total_power_apparent_lag_5m', 'total_power_apparent_lag_1h', 'total_power_apparent_lag_24h', 'total_power_apparent_lag_168h', 'import_energy_active_lag_5m', 'import_energy_active_lag_1h', 'import_energy_active_lag_24h', 'import_energy_active_lag_168h', 'export_energy_active_lag_5m', 'export_energy_active_lag_1h', 'export_energy_active_lag_24h', 'export_energy_active_lag_168h', 'import_energy_reactive_lag_5m', 'import_energy_reactive_lag_1h', 'import_energy_reactive_lag_24h', 'import_energy_reactive_lag_168h', 'export_energy_reactive_lag_5m', 'export_energy_reactive_lag_1h', 'export_energy_reactive_lag_24h', 'export_energy_reactive_lag_168h', 'total_energy_active_lag_5m', 'total_energy_active_lag_1h', 'total_energy_active_lag_24h', 'total_energy_active_lag_168h', 'total_energy_reactive_lag_5m', 'total_energy_reactive_lag_1h', 'total_energy_reactive_lag_24h', 'total_energy_reactive_lag_168h']
    )

In [ ]:
df2 = pd.DataFrame(
    train[1][0],
    columns=['time', 'device_id', 'hour', 'weekday', 'month', 'total_power_active_lag_5m', 'total_power_active_lag_1h', 'total_power_active_lag_24h', 'total_power_active_lag_168h', 'total_power_reactive_lag_5m', 'total_power_reactive_lag_1h', 'total_power_reactive_lag_24h', 'total_power_reactive_lag_168h', 'total_power_apparent_lag_5m', 'total_power_apparent_lag_1h', 'total_power_apparent_lag_24h', 'total_power_apparent_lag_168h', 'import_energy_active_lag_5m', 'import_energy_active_lag_1h', 'import_energy_active_lag_24h', 'import_energy_active_lag_168h', 'export_energy_active_lag_5m', 'export_energy_active_lag_1h', 'export_energy_active_lag_24h', 'export_energy_active_lag_168h', 'import_energy_reactive_lag_5m', 'import_energy_reactive_lag_1h', 'import_energy_reactive_lag_24h', 'import_energy_reactive_lag_168h', 'export_energy_reactive_lag_5m', 'export_energy_reactive_lag_1h', 'export_energy_reactive_lag_24h', 'export_energy_reactive_lag_168h', 'total_energy_active_lag_5m', 'total_energy_active_lag_1h', 'total_energy_active_lag_24h', 'total_energy_active_lag_168h', 'total_energy_reactive_lag_5m', 'total_energy_reactive_lag_1h', 'total_energy_reactive_lag_24h', 'total_energy_reactive_lag_168h']
    )

In [ ]:
df1.shape, df2.shape

((273488, 41), (557104, 41))

In [ ]:
print(df1.device_id.value_counts())
df2.device_id.value_counts()

4ad3e3e6-5f11-4923-a88d-668b4ddb493a    7059
478e1210-ead4-4d55-9d35-c1090fbeb354    7052
3f3f551c-6594-4098-b9b4-af9ac4ab606c    7051
69a3bfa0-e8e6-41be-8c42-65e899a37794    7043
8f361e91-54a8-4f05-bab5-a672fc9d82b6    7035
114b44b0-0dc4-45d6-a4ac-83223ea9b2d8    7034
dd40019b-5b35-4fba-b0dd-16083c1fdb84    7032
2c55f2fe-8dfd-4fb2-a838-d5717b9f1488    7030
b14559e0-1c2e-417e-a736-f92a39bec0db    7027
ea9ac4b4-00e8-4ce2-a211-db429fb81cde    7027
fca96b30-ca2f-41b8-b987-3207af296b03    7027
a2aee527-f5e2-41c1-9da0-6c4fd1b1d4ba    7023
8e0a1d6f-8ba6-4d0c-a025-378ace0f7291    7021
51c62c18-e9e5-4aa9-b9ec-a8a2a2150583    7020
615c3df8-9ef6-4aa1-994a-87c2a7ffa0cc    7019
522e56a5-ff0c-4ece-923f-a925aabe15e6    7019
62c4699f-2e30-494b-a945-0801ccaf7bf9    7017
4a448440-d2dd-4b27-ae02-87e7e3baa9f5    7017
84faf80e-e9d5-4552-b919-fbb8df7c7e4a    7015
12db0eaf-e9e5-4cb5-849c-3289ef475f02    7014
8f3a5182-51f7-4a43-a580-f272cd1c4481    7013
1540b469-3e7e-4e8e-89fb-a1aceaf8dbd6    7011
265064b5-7

4ad3e3e6-5f11-4923-a88d-668b4ddb493a    14362
478e1210-ead4-4d55-9d35-c1090fbeb354    14358
3f3f551c-6594-4098-b9b4-af9ac4ab606c    14350
2c55f2fe-8dfd-4fb2-a838-d5717b9f1488    14350
69a3bfa0-e8e6-41be-8c42-65e899a37794    14346
dd40019b-5b35-4fba-b0dd-16083c1fdb84    14341
b14559e0-1c2e-417e-a736-f92a39bec0db    14341
fca96b30-ca2f-41b8-b987-3207af296b03    14338
ea9ac4b4-00e8-4ce2-a211-db429fb81cde    14337
62c4699f-2e30-494b-a945-0801ccaf7bf9    14336
615c3df8-9ef6-4aa1-994a-87c2a7ffa0cc    14336
114b44b0-0dc4-45d6-a4ac-83223ea9b2d8    14335
8f361e91-54a8-4f05-bab5-a672fc9d82b6    14333
8e0a1d6f-8ba6-4d0c-a025-378ace0f7291    14323
51c62c18-e9e5-4aa9-b9ec-a8a2a2150583    14319
265064b5-79b8-4436-8bc7-3f24062d73c3    14312
a2aee527-f5e2-41c1-9da0-6c4fd1b1d4ba    14300
522e56a5-ff0c-4ece-923f-a925aabe15e6    14298
12db0eaf-e9e5-4cb5-849c-3289ef475f02    14294
f90e7387-9810-408d-885b-6f3841f4ec3f    14290
4a448440-d2dd-4b27-ae02-87e7e3baa9f5    14288
1540b469-3e7e-4e8e-89fb-a1aceaf8db

In [ ]:
print(train[0][0][:,0].min(),'|', train[0][0][:,0].max())
print(train[1][0][:,0].min(), '|', train[1][0][:,0].max())

2024-01-07 23:00:02.287000 | 2024-02-03 07:56:09.038000
2024-01-07 23:00:02.287000 | 2024-03-01 19:10:32.924000


In [ ]:
print(valid[0][0][:,0].min(),'|', valid[0][0][:,0].max())
print(valid[1][0][:,0].min(), '|', valid[1][0][:,0].max())

2024-02-03 07:56:10.841000 | 2024-03-01 19:10:32.924000
2024-03-01 19:10:34.727000 | 2024-03-29 08:20:18.081000


In [ ]:
for split in range(len(train)):
    ts_index = train[split][0][:,0]
    print(f"split {split}", ts_index.min(), ts_index.max())
    time.sleep(1)

split 0 2024-01-07 23:00:02.287000 2024-02-03 07:56:09.038000
split 1 2024-01-07 23:00:02.287000 2024-03-01 19:10:32.924000
split 1 2024-01-07 23:00:02.287000 2024-03-01 19:10:32.924000
split 2 2024-01-07 23:00:02.287000 2024-03-29 08:20:18.081000
split 2 2024-01-07 23:00:02.287000 2024-03-29 08:20:18.081000
split 3 2024-01-07 23:00:02.287000 2024-04-25 15:45:48.977000
split 3 2024-01-07 23:00:02.287000 2024-04-25 15:45:48.977000
split 4 2024-01-07 23:00:02.287000 2024-05-22 18:20:29.223000
split 4 2024-01-07 23:00:02.287000 2024-05-22 18:20:29.223000
split 5 2024-01-07 23:00:02.287000 2024-06-18 11:15:26.027000
split 5 2024-01-07 23:00:02.287000 2024-06-18 11:15:26.027000
split 6 2024-01-07 23:00:02.287000 2024-07-15 04:40:52.171000
split 6 2024-01-07 23:00:02.287000 2024-07-15 04:40:52.171000
split 7 2024-01-07 23:00:02.287000 2024-08-12 04:00:08.678000
split 7 2024-01-07 23:00:02.287000 2024-08-12 04:00:08.678000
split 8 2024-01-07 23:00:02.287000 2024-09-09 11:10:03.281000
split 8 

## Example model training using the loaded data

In [ ]:
grace_period = 0
split_confidence = 0.1

Fodt_5 = fastodt.FastODT(
        depth=20,
        grace_period=grace_period,
        split_confidence=split_confidence,
        option='cb'
    )

In [ ]:
# Use the first split of the training data
if isinstance(train, list):
    # For walk-forward validation structure
    X, y = train[0]

    # Filter for a specific device if needed
    mask = X[:, 1] == "231e45ef-ca7a-4f80-be11-ac7e2da0274c"
    X_device = X[mask]
    y_device = y[mask]

    # Train on a subset
    n_samples = min(200, len(X_device))
    Fodt_5.partial_fit(X_device[:n_samples, 2:], y_device[:n_samples])
else:
    # For single tuple structure
    X, y = train
    mask = X[:, 1] == "231e45ef-ca7a-4f80-be11-ac7e2da0274c"
    X_device = X[mask]
    y_device = y[mask]

    # Train on a subset
    n_samples = min(200, len(X_device))
    Fodt_5.partial_fit(X_device[:n_samples, 2:], y_device[:n_samples])

In [ ]:
# Generate predictions
Y_pred = np.array([Fodt_5.predict(x) for x in X_device[:n_samples, 2:]])